In [9]:
import numpy as np
import random
import pandas as pd
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8" # set before import torch
import torch
torch.use_deterministic_algorithms(True)

import torch.nn as nn
import torch.distributed as dist
from transformers import AutoTokenizer, AutoModel
from transformers import Trainer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from safetensors.torch import load_file
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr, pearsonr
# import wandb

def set_seed(seed):
    random.seed(seed)                     # Python RNG
    np.random.seed(seed)                  # NumPy RNG
    torch.manual_seed(seed)               # CPU RNG
    torch.cuda.manual_seed_all(seed)      # All GPU RNGs
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)  # Enforce deterministic ops (PyTorch ≥ 1.8)

set_seed(91)

#data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

class ExpWeightedMSELoss(nn.Module):
    def __init__(self, alpha=1, ignore_index=-100):
        super().__init__()
        self.alpha = alpha
        self.ignore_index = ignore_index

    def forward(self, y_pred, y_true):
        mask = (y_true != self.ignore_index)
        if mask.any():
            y_pred = y_pred[mask]
            y_true = y_true[mask]
            weights = torch.exp(self.alpha * y_true)
#            skipped = (~mask).sum().item()
#            print(f"[Loss] Skipped {skipped} invalid labels")
            return torch.mean(weights * (y_pred - y_true) ** 2)
        else:
            # No valid labels in batch, return safe zero loss
            return (y_pred - y_pred).sum() * 0.0

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states):
        # hidden_states: (batch, seq_len, hidden_size)
        attn_weights = torch.softmax(self.attn(hidden_states), dim=1)  # (batch, seq_len, 1)
        pooled = (attn_weights * hidden_states).sum(dim=1)  # (batch, hidden_size)
        return pooled

class AttentionPooling2(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states, mask=None, input_ids=None, cls_token_id=None, eos_token_id=None, return_weights=False):
        # hidden_states: (batch, seq_len, hidden_size)
        attn_scores = self.attn(hidden_states).squeeze(-1)  # (batch, seq_len)

        if mask is not None:
            # Optionally zero out CLS and EOS tokens by updating the mask
            if input_ids is not None and cls_token_id is not None and eos_token_id is not None:
                cls_mask = (input_ids == cls_token_id)
                eos_mask = (input_ids == eos_token_id)
                ignore_mask = cls_mask | eos_mask
                mask = mask.masked_fill(ignore_mask, 0)

            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = torch.softmax(attn_scores, dim=1)
        pooled = torch.sum(hidden_states * attn_weights.unsqueeze(-1), dim=1)
        if return_weights:
            return pooled, attn_weights
        else:
            return pooled

class ESM2Regressor(nn.Module):
    def __init__(self, base_model, head_type="linear"):
        super().__init__()
        self.esm = base_model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.cls_token_id = self.tokenizer.cls_token_id
        self.eos_token_id = self.tokenizer.eos_token_id
        
        hidden_size = base_model.config.hidden_size
        self.attn_pool = AttentionPooling(hidden_size)
        self.attn_pool2 = AttentionPooling2(hidden_size)
        self.head_type = head_type.lower()
        self.dropout = nn.Dropout(0.1)

        if self.head_type == "linear":
            self.head = nn.Linear(hidden_size, 1)

        elif self.head_type == "mlp_relu":
            self.head = nn.Sequential(
                nn.Linear(hidden_size, hidden_size),
                nn.ReLU(),
#                nn.Dropout(0.1),
                nn.Linear(hidden_size, 1)
            )
            
        elif self.head_type == "dense":
            self.head = nn.Sequential(
                nn.Linear(hidden_size, hidden_size),
                nn.Linear(hidden_size, 1)
            )

        elif self.head_type == "mlp_gelu":
            self.head = nn.Sequential(
                nn.Linear(hidden_size, hidden_size),
                nn.GELU(),
#                nn.Dropout(0.1),
                nn.Linear(hidden_size, 1)
            )

        elif self.head_type == "residual_mlp":
            self.head = nn.Sequential(
                nn.Linear(hidden_size, 1920),
                nn.GELU(),
                nn.Linear(1920, hidden_size),
                nn.GELU(),
                nn.Linear(hidden_size, 1)
            )

        elif self.head_type == "attention_pooling":
            self.head = nn.Linear(hidden_size, 1)

        elif self.head_type == "attention_pooling2":
            self.head = nn.Linear(hidden_size, 1)

        elif self.head_type == "transformer_head":
            encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=8, batch_first=True)
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
            self.head = nn.Linear(hidden_size, 1)

        else:
            raise ValueError(f"Unsupported head_type: {self.head_type}")

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.esm(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state  # (batch, seq_len, hidden_size)

        return_attn_weights = (labels is None)

        if self.head_type in ["linear", "dense", "mlp_relu", "mlp_gelu", "residual_mlp"]:
            x = self.attn_pool2(hidden_states = hidden, 
                               mask=attention_mask, 
                               input_ids=input_ids, 
                               cls_token_id=self.cls_token_id, 
                               eos_token_id=self.eos_token_id,
                               return_weights=return_attn_weights)
#            x = self.dropout(x)

        elif self.head_type == "attention_pooling":
            x = self.attn_pool(hidden)
#            x = self.dropout(x)
        
        elif self.head_type == "attention_pooling2":
            x = self.attn_pool2(hidden_states = hidden, 
                               mask=attention_mask, 
                               input_ids=input_ids, 
                               cls_token_id=self.cls_token_id, 
                               eos_token_id=self.eos_token_id,
                               return_weights=return_attn_weights)
#            x = self.dropout(x)

        elif self.head_type == "transformer_head":
            transformed = self.transformer(hidden)
            x = self.attn_pool2(hidden_states = transformed, 
                               mask=attention_mask, 
                               input_ids=input_ids, 
                               cls_token_id=self.cls_token_id, 
                               eos_token_id=self.eos_token_id,
                               return_weights=return_attn_weights)
#            x = self.dropout(cls_token)

        else:
            raise RuntimeError("Unsupported head type in forward pass.")

        if isinstance(x, tuple):
            x, attn_weights = x
        else:
            attn_weights = None
        logits = self.head(x).squeeze(-1)
        loss = None
        if labels is not None:
            if not torch.is_tensor(labels):
                labels = torch.tensor(labels, dtype=torch.float, device=logits.device)
            else:
                labels = labels.float().to(logits.device)      

            loss_fn = nn.MSELoss()
            loss = loss_fn(logits, labels)
            return {"loss": loss, "logits": logits}
        else:
            return {"logits": logits, "outputs": outputs, "attn_pool2_weights": attn_weights}
        

def get_layerwise_lr_params(model, base_lr=2e-5, decay=0.95):
    no_decay = ["bias", "LayerNorm.weight"]
    grouped_params = []

    encoder_layers = model.esm.encoder.layer
    num_layers = len(encoder_layers)

    for i, layer in enumerate(encoder_layers):
        lr = base_lr * (decay ** (num_layers - i - 1))

        grouped_params.append({
            "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": 0.01,
            "lr": lr,
        })
        grouped_params.append({
            "params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": lr,
        })

    # Add ESM embeddings (optional)
    grouped_params.append({
        "params": model.esm.embeddings.parameters(),
        "lr": base_lr * (decay ** num_layers)
    })

    # Add custom head
    grouped_params.append({
        "params": model.head.parameters(),  # or model.regressor, etc.
        "lr": base_lr,
        "weight_decay": 0.01,
    })

    return grouped_params

class LLRDTrainer(Trainer):
    def create_optimizer(self):
        if self.optimizer is None:
            optimizer_grouped_parameters = get_layerwise_lr_params(self.model, base_lr=self.args.learning_rate)
            self.optimizer = torch.optim.AdamW(
                optimizer_grouped_parameters,
                lr=self.args.learning_rate,
                betas=(0.9, 0.999),
                eps=1e-8,
            )
        return self.optimizer



head_type="linear"
# model_name = "./Genesis_quant_v18_8k_linear_padding_head_benchmarking/checkpoint-3614"
model_name = "facebook/esm2_t12_35M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)
model = ESM2Regressor(base_model, head_type=head_type)
# attn2
state_dict = load_file("./Genesis_quant_v18_8k_linear_padding_head_benchmarking/checkpoint-3614/model.safetensors") 
# state_dict = load_file("./8k_head_benchmark2_attention_pooling2/checkpoint-1807/model.safetensors")
model.load_state_dict(state_dict)
model.eval()

train_sequences, test_sequences, train_labels, test_labels, train_accessions, test_accessions = train_test_split(sequences, labels, accessions, test_size=0.10, shuffle=True, random_state=91)

test_sequences_plus101 = test_sequences + test2_sequences
test_labels_plus101 = test_labels + test2_labels

train_tokenized = tokenizer(train_sequences)
test_tokenized = tokenizer(test_sequences_plus101)

from datasets import Dataset
train_dataset = Dataset.from_dict(train_tokenized)
test_dataset = Dataset.from_dict(test_tokenized)

train_dataset = train_dataset.add_column("labels", train_labels)
test_dataset = test_dataset.add_column("labels", test_labels_plus101)
train_dataset

print("Let's use", torch.cuda.device_count(), "GPUs!")

/home/zf77/.conda/envs/torch/lib/python3.11/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Let's use 0 GPUs!


/home/zf77/.conda/envs/torch/lib/python3.11/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [10]:

inputs = all_datasets["hTMC_sequence"].tolist()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_tokenized = tokenizer(
    inputs,
    padding=True,             # pad shorter sequences to the length of the longest one
    truncation=True,          # truncate longer sequences if they exceed model max length
    return_tensors="pt"       # return PyTorch tensors
).to(device)

model.to(device)
model.eval()
W = model.head.weight.squeeze(0).to(device)  # shape: (hidden_size,)
b = model.head.bias.item()

batch_size = 4
# Function to process in batches
def process_in_batches(inputs, model, batch_size):
    all_outputs = []
    all_hidden_states = []
    all_logits = []
    all_attn2_weights = []

    for i in range(0, len(inputs["input_ids"]), batch_size):
        # Prepare the batch inputs
        batch = {k: v[i:i + batch_size].to(device) for k, v in inputs.items()}

        # Perform inference
        with torch.no_grad():
            results = model(**batch)
            attn2_weights = results["attn_pool2_weights"].to(device)
            logits = results["logits"].to(device)
            outputs = results["outputs"]
            all_attn2_weights.append(attn2_weights)          
            all_logits.append(logits)
            all_outputs.append(outputs)
    
    all_logits = torch.cat(all_logits, dim=0)
    all_hidden_states = [output.last_hidden_state for output in all_outputs]
    all_hidden_states = torch.cat(all_hidden_states, dim=0)
    all_attn2_weights = torch.cat(all_attn2_weights, dim=0)

    return all_logits, all_hidden_states, all_attn2_weights

all_logits, all_hidden_states, all_attn2_weights = process_in_batches(test_tokenized, model, batch_size)



